In [1]:
import os
from scipy import signal
import nibabel as nib
import numpy as np
import argparse
from Unet import Generic_UNet, InitWeights_He
import math
from tqdm import tqdm
import torch
import torch.nn as nn
from scipy.ndimage.filters import gaussian_filter
from collections import OrderedDict
from numpy.linalg import inv
from utilities import SoftDiceLoss, resize_segmentation, _concat, _concatmodel, ComputMetric
from sampling_multiprocess import get_augment_par
from common_test_Unet import pad_nd_image, _compute_steps_for_sliding_window, getallbatch
import torch.nn.functional as F

/tmp/ipykernel_3018984/3809002401.py:11: DeprecationWarning: Please use `gaussian_filter` from the `scipy.ndimage` namespace, the `scipy.ndimage.filters` namespace is deprecated.
  from scipy.ndimage.filters import gaussian_filter


## Test performance of meningioma

In [2]:
from common_test_Unet import tta_rolling
def nntestMeningioma(model, saveresults, name, trainval = False, ImgsegmentSize = [128, 128, 128], deepsupervision = False, DatafileValFold=None, tta=False, ttalist = [0], ttalistprob=[1], NumsClass = 2, channel = 1):
    batch_size = 1
    NumsInputChannel = 1
    if trainval == False:
        DatafileFold = DatafileValFold
        DatafileImgc1 = DatafileFold + 'Imgpre-eval.txt'
        DatafileLabel = DatafileFold + 'seg-eval.txt'
    else:
        DatafileFold = DatafileValFold
        DatafileImgc1 = DatafileFold + 'Imgpre-train.txt'
        DatafileLabel = DatafileFold + 'seg-train.txt'

    Imgfilec1 = open(DatafileImgc1)
    Imgreadc1 = Imgfilec1.read().splitlines()
    Labelfile = open(DatafileLabel)
    Labelread = Labelfile.read().splitlines()

    DSClist = []
    SENSlist = []
    PREClist = []
    PredSumlist = []

    for numr in range(len(Imgreadc1)):
        # for numr in range(10, 11):

        Imgnamec1 = Imgreadc1[numr]
        Imgloadc1 = nib.load(Imgnamec1)
        Imgc1 = Imgloadc1.get_fdata()
        if channel > 1:
            # for flair and dwi
            Imgloadc2 = nib.load(Imgnamec1.replace('image.nii.gz', 'image_c2.nii.gz'))
            Imgc2 = Imgloadc2.get_fdata()
            channels = np.stack((Imgc1, Imgc2), axis = 0)
        else:
            channels = Imgc1[None, ...] ## add one dimension
        Labelname = Labelread[numr]
        Labelload = nib.load(Labelname)
        gtlabel = Labelload.get_fdata()

        knamelist = Imgnamec1.split("/")
        kname = knamelist[-2]

        hp_results = tta_rolling(model, channels, batch_size, ImgsegmentSize, NumsInputChannel, NumsClass, tta, ttalist, ttalistprob, deepsupervision)

        predSegmentation = np.argmax(hp_results, axis=0)
        ## use the mask to constratin the results
        PredSegmentationWithinRoi = predSegmentation
        # PredSegmentationWithinRoi = predSegmentation
        # sio.savemat('./result.mat', {'results': PredSegmentationWithinRoi})
        imgToSave = PredSegmentationWithinRoi

        if saveresults:
            npDtype = np.dtype(np.float32)
            proxy_origin = nib.load(Imgnamec1)
            hdr_origin = proxy_origin.header
            affine_origin = proxy_origin.affine
            proxy_origin.uncache()

            newLabelImg = nib.Nifti1Image(imgToSave, affine_origin)
            newLabelImg.set_data_dtype(npDtype)

            dimsImgToSave = len(imgToSave.shape)
            newZooms = list(hdr_origin.get_zooms()[:dimsImgToSave])
            if len(newZooms) < dimsImgToSave:  # Eg if original image was 3D, but I need to save a multi-channel image.
                newZooms = newZooms + [1.0] * (dimsImgToSave - len(newZooms))
            newLabelImg.header.set_zooms(newZooms)

            directory = "./output/Meningioma/%s/" % (name)
            if not os.path.exists(directory):
                os.makedirs(directory)
            savename = directory + 'pred_' + kname + '_Segm.nii.gz'
            nib.save(newLabelImg, savename)

        labelc1 = gtlabel == 1
        predc1 = imgToSave == 1

        DSCc1, SENSc1, PRECc1 = ComputMetric(labelc1, predc1)

        DSClist.append([DSCc1])
        SENSlist.append([SENSc1])
        PREClist.append([PRECc1])
        print('case ' + str(numr) + ' done, DSC is ' + str(DSCc1))
        print('total GT pixel is ' + str(np.sum(labelc1)))
        print('total predicted pixel is ' + str(np.sum(predc1)))

    DSClist = np.array(DSClist)
    DSCmean = DSClist.mean(axis=0)
    SENSlist = np.array(SENSlist)
    SENSmean = SENSlist.mean(axis=0)
    PREClist = np.array(PREClist)
    PRECmean = PREClist.mean(axis=0)

    return DSCmean, SENSmean, PRECmean

In [3]:
# single modality
# meningioimackpt = './output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20/checkpoint.pth.tar'
meningioimackpt = './output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20/model_best.pth.tar'
# multiple modality.
meningioimackpt = './output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20_c2/model_best.pth.tar'

meningioimapath = './data/datafile/Dataset_Meningioma/val/'

In [4]:
patch_size = [320, 320, 20]
# patch_size = [256, 256, 32]
testlist = [0]
testprob = [1]
NumsInputChannel = 2
NumsClass = 2
DatafileValFoldtr = None
DatafileValFoldts = meningioimapath
#
args = {'resume': meningioimackpt, 'saveresults': True, 'patch_size': patch_size, 'downsampling': 4,
       'ttalist': testlist, 'ttalistprob': testprob, 'features': 30, 'deepsupervision': True, 'gpu': 0}

In [5]:
# create model
conv_op = nn.Conv3d
dropout_op = nn.Dropout3d
norm_op = nn.InstanceNorm3d
conv_per_stage = 2
base_num_features = args['features']
args['features'] = base_num_features

norm_op_kwargs = {'eps': 1e-5, 'affine': True}
dropout_op_kwargs = {'p': 0, 'inplace': True}
net_nonlin = nn.LeakyReLU
net_nonlin_kwargs = {'negative_slope': 1e-2, 'inplace': True}
net_num_pool_op_kernel_sizes = []
for kiter in range(0, args['downsampling']):  # (0,5)
    net_num_pool_op_kernel_sizes.append([2, 2, 1])
# for kiter in range(0, args['downsampling'] - 1):  # (0,5)
#     net_num_pool_op_kernel_sizes.append([2, 2, 1])
# net_num_pool_op_kernel_sizes.append([2, 2, 2])
net_conv_kernel_sizes = []
for kiter in range(0, args['downsampling'] + 1):  # (0,6)
    net_conv_kernel_sizes.append([3, 3, 3])

model = Generic_UNet(NumsInputChannel, base_num_features, NumsClass,
                     len(net_num_pool_op_kernel_sizes),
                     conv_per_stage, 2, conv_op, norm_op, norm_op_kwargs, dropout_op,
                     dropout_op_kwargs,
                     net_nonlin, net_nonlin_kwargs, args['deepsupervision'], False, lambda x: x, InitWeights_He(1e-2),
                     net_num_pool_op_kernel_sizes, net_conv_kernel_sizes, False, True, True)
model = model.cuda()
model.eval()

Generic_UNet(
  (conv_blocks_localization): ModuleList(
    (0): Sequential(
      (0): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin(
            (conv): Conv3d(480, 240, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (instnorm): InstanceNorm3d(240, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
          )
        )
      )
      (1): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin(
            (conv): Conv3d(240, 240, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (instnorm): InstanceNorm3d(240, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
          )
        )
      )
    )
    (1): Sequential(
      (0): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin

In [6]:
torch.cuda.set_device(args['gpu'])
if args['resume']:
    if os.path.isfile(args['resume']):
        print("=> loading checkpoint '{}'".format(args['resume']))
        checkpoint = torch.load(args['resume'], map_location='cuda:' + str(args['gpu']))
        model.load_state_dict(checkpoint['state_dict'])
        print("=> loaded checkpoint '{}' (epoch {})".format(args['resume'], checkpoint['epoch']))
    else:
        print("=> no checkpoint found at '{}'".format(args['resume']))

=> loading checkpoint './output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20_c2/model_best.pth.tar'
=> loaded checkpoint './output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20_c2/model_best.pth.tar' (epoch 601)


In [8]:
DSC, SENS, PREC = nntestMeningioma(model, True, 'fomo25val/results/', False,
                        ImgsegmentSize=args["patch_size"], 
                        deepsupervision=args["deepsupervision"], DatafileValFold=DatafileValFoldts, channel = NumsInputChannel)

case 0 done, DSC is 0
total GT pixel is 12007
total predicted pixel is 59
case 1 done, DSC is 0.4240552995391705
total GT pixel is 16412
total predicted pixel is 5288
case 2 done, DSC is 0.791162759273022
total GT pixel is 8070
total predicted pixel is 6731
case 3 done, DSC is 0.3367983367983368
total GT pixel is 263
total predicted pixel is 699
case 4 done, DSC is 0
total GT pixel is 742
total predicted pixel is 4476


In [9]:
print(DSC)
print(SENS)
print(PREC)

[0.31040328]
[0.32436797]
[0.39433975]
